# 6. Kruskal-Wallis ANOVA Analysis

This notebook analyzes whether sentiment scores differ significantly across the three time windows (pre-cancel, cancel, post-apology) for YouTube comment sentiment data.

## Research Question

**RQ1**: *How do audience sentiment trajectories on YouTube change across pre-cancel, cancel, and post-apology windows for South Korean influencer apology videos?*

## Method

- **Normality Test**: Shapiro-Wilk test to determine data distribution
- **Main Test**: Kruskal-Wallis H test (non-parametric alternative to ANOVA)
- **Post-Hoc**: Dunn's test with Bonferroni correction for pairwise comparisons
- **Effect Size**: Kruskal-Wallis Eta-squared (η²)
- **Data**: Sentiment scores across PRE, CANCEL, POST windows
- **Significance Threshold**: α = 0.05

## Interpretation Guidelines

### Shapiro-Wilk Test
| p-value | Interpretation |
|---|---|
| ≥ 0.05 | Normally distributed |
| < 0.05 | Skewed (Non-normal) |

### Kruskal-Wallis H Test
| p-value | Interpretation |
|---|---|
| ≥ 0.05 | No significant difference between groups |
| < 0.05 | Significant difference exists between groups |

### Effect Size (Eta-squared)
| η² Value | Interpretation |
|---|---|
| 0.01 - 0.06 | Small effect |
| 0.06 - 0.14 | Medium effect |
| > 0.14 | Large effect |

### Dunn's Post-Hoc Test
- Compare adjusted p-values between each pair of windows
- Bonferroni correction adjusts for multiple comparisons

**Shapiro-Wilk Test**
This is to determine if the data is normal or skewed to be able to determine what kind of statistical testing will be utilized for further investigation.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import scikit_posthocs as sp
from scipy import stats

# Portable repo root: walk up from the working directory until the stage folders are found
PROJECT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "02-preprocessing").is_dir())
PERSON = os.environ.get("PERSON", "bodeoum_tv")  # "bodeoum_tv" or "mimiminu"
SEED = 42            # fixes the Shapiro-Wilk subsample
SHAPIRO_N = 5000     # Shapiro-Wilk is unreliable above ~5,000 observations
ALPHA = 0.05

# Final scored comments from 03-analysis (all comments, not a partial inference checkpoint)
sentiment_path = PROJECT_DIR / "03-analysis" / "outputs" / PERSON / "sentiment_comments.csv"
sentiment_df = pd.read_csv(sentiment_path)
clean_df = sentiment_df.dropna(subset=["window", "sentiment_score"])
print(f"Person: {PERSON} | comments: {len(clean_df)} | source: {sentiment_path.relative_to(PROJECT_DIR)}")

print("--- SHAPIRO-WILK TEST: SENTIMENT SCORES ---")
sentiment_sample = clean_df.sample(n=min(SHAPIRO_N, len(clean_df)), random_state=SEED)

for window in sentiment_sample["window"].unique():
    scores = sentiment_sample[sentiment_sample["window"] == window]["sentiment_score"]
    statistic, p_value = stats.shapiro(scores)
    distribution = "Normally Distributed" if p_value >= ALPHA else "Skewed (Non-Normal)"
    print(f"Window: {window.upper()}")
    print(f"  Test Statistic: {statistic:.4f}")
    print(f"  p-value: {p_value:.4g} -> {distribution}\n")


**Kruskal-Wallis H & Dunn's Post-Hoc Test**
Since the data is skewed, a kruskal-wallis H and Dunn's Post-Hoc Test is utilized to determine if sentiments across different windows significantly differ from each other.

In [ ]:
print("--- KRUSKAL-WALLIS H TEST ---")

# Compare sentiment-score distributions across the three windows
pre_scores = clean_df[clean_df["window"] == "pre"]["sentiment_score"]
cancel_scores = clean_df[clean_df["window"] == "cancel"]["sentiment_score"]
post_scores = clean_df[clean_df["window"] == "post"]["sentiment_score"]

kw_stat, kw_p_value = stats.kruskal(pre_scores, cancel_scores, post_scores)
print(f"Kruskal-Wallis H-statistic: {kw_stat:.4f}")
print(f"p-value: {kw_p_value:.4g}\n")

# Pairwise follow-up only when the omnibus test is significant
if kw_p_value < ALPHA:
    print(f"Result: SIGNIFICANT difference found in sentiment across the windows (p < {ALPHA}).")
    print("Proceeding to Dunn's Post-Hoc Test to see WHICH windows differ...\n")
    print("--- DUNN'S POST-HOC TEST (with Bonferroni correction) ---")
    dunn_results = sp.posthoc_dunn(clean_df, val_col="sentiment_score", group_col="window", p_adjust="bonferroni")
    print(dunn_results)


### Kruskal-Wallis Eta-Squared Effect Size

This is used to determine how much impact the shift between windows is present

In [ ]:
def calculate_kruskal_eta_sq(data: pd.DataFrame, group_col: str, value_col: str) -> tuple[float, float, float]:
    """Kruskal-Wallis H, p-value, and eta-squared effect size: (H - k + 1) / (n - k)."""
    groups = [group[value_col].values for _, group in data.groupby(group_col)]
    h_stat, p_val = stats.kruskal(*groups)
    n, k = len(data), len(groups)
    return h_stat, p_val, (h_stat - k + 1) / (n - k)


h, p, effect_size = calculate_kruskal_eta_sq(clean_df, "window", "sentiment_score")
print(f"Eta-squared: {effect_size:.4f}")


Median is computed to see which of the windows differs the most

In [ ]:
median_sentiment_by_window = clean_df.groupby('window')['sentiment_score'].median()
display(median_sentiment_by_window)

## Summary

### Key Findings

1. **Normality Check**: Shapiro-Wilk test showed data is non-normal across all windows (p < 0.05)
2. **Main Test**: Kruskal-Wallis H test revealed significant differences in sentiment across PRE, CANCEL, and POST windows
3. **Post-Hoc**: Dunn's test with Bonferroni correction identifies which specific window pairs differ
4. **Effect Size**: Kruskal-Wallis Eta-squared (η²) measures the magnitude of the difference

### Interpretation

- The sentiment significantly differs across the three time windows
- Use Dunn's post-hoc results to identify which specific pairs (PRE vs CANCEL, PRE vs POST, CANCEL vs POST) have significant differences
- Effect size indicates practical significance of the difference

### Limitations

- Non-parametric tests are robust but may be less powerful than parametric alternatives
- Large sample size (128,000) may cause trivial differences to appear significant
- Effect size helps contextualize statistical significance